In [ ]:
# STEP 1 - Generate external knowledge base
# pip install pypdf cryptography faiss-cpu langchain_community chromadb gradio

from langchain_community.document_loaders import PyPDFLoader

#loader = PyPDFLoader("C://Anand//old_laptop_backup//D drive data//Anand//material//Training//Gen_AI//L2_Applied_Gen_AI//external_knowledgebase_for_rag//Python_Programming.pdf")  # Upload your .pdf
loader = PyPDFLoader("C://Anand//old_laptop_backup//D drive data//Anand//material//Training//Gen_AI//L2_Applied_Gen_AI//external_knowledgebase_for_rag//Bhagavad-gita_As_It_Is english.pdf")  # Upload your .pdf
pages = loader.load()

print("📄 Page 1 Content:\n", pages[0].page_content[:500])
print("🗂️ Metadata:\n", pages[0].metadata)


In [ ]:
# STEP 2 - Split the external knowledge

from langchain_text_splitters import RecursiveCharacterTextSplitter

# Split the data
splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50,
    separators=["\n\n", "\n", ".", " ", ""]
)

split_docs = splitter.split_documents(pages)

print(f"🔹 Total Chunks: {len(split_docs)}")
print("📄 Sample Chunk:\n", split_docs[0].page_content)

In [ ]:
# STEP 3 - Create embeddings & vector stores with FAISS
from langchain_ollama import OllamaEmbeddings, ChatOllama
from langchain_community.vectorstores import FAISS
from dotenv import find_dotenv, load_dotenv
import os

env_path = find_dotenv()
if not env_path:
    raise FileNotFoundError(".env file not found.")

load_dotenv(env_path)
apiKey = os.getenv("OLLAMA_API_KEY")
embedding_model = OllamaEmbeddings(model="nomic-embed-text", api_key=apiKey)
vector_store = FAISS.from_documents(split_docs, embedding=embedding_model)
print(vector_store)

In [ ]:
# STEP 3 - Create embeddings & vector stores using Chroma

from langchain_community.vectorstores import Chroma

# Define the directory to store ChromaDB data
persist_directory = "c:/content/chroma_db"

# Create a Chroma vector store from the text chunks and embeddings, and persist it to disk
vector_store = Chroma.from_documents(
    documents=split_docs,
    embedding=embedding_model,
    persist_directory=persist_directory
)
print(type(vector_store))
print(f"Embeddings successfully stored in ChromaDB and persisted to {persist_directory}.")
print(vector_store)

In [ ]:
# STEP 4 - Integrate with LLM & build retrieval chain
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_classic.chains import RetrievalQA
custom_prompt_template = """Use the following pieces of context to answer the user's question.
If you don't know the answer, just say that you don't know, don't try to make up an answer.
--------------------
{context}
Question: {question}
"""
CUSTOM_PROMPT = PromptTemplate(
    template=custom_prompt_template, input_variables=["context", "question"]
)
llm = ChatOllama(model="gpt-oss:120b-cloud")
parser = StrOutputParser()
qa_chain = RetrievalQA.from_chain_type(
    llm,
    chain_type="stuff", # Other chain types include "map_reduce", "refine", "map_rerank"
    retriever=vector_store.as_retriever(),
    return_source_documents=True, # Optional: return the retrieved documents
    chain_type_kwargs={"prompt": CUSTOM_PROMPT} # Pass the custom prompt here
)



In [ ]:
# STEP 5 - CLI interface

while True:
    question = input("\nEnter the question (or type 'exit' to quit): ")

    if question.lower() in ['exit', 'quit']:
        print("Exiting... Have a great day!")
        break

    response = qa_chain({"query": question})
    answer = response["result"]
    source_documents = response["source_documents"]

    # Print the response
    print(f"Bot: {answer}")
    print(f"Source documents: {source_documents}")


In [ ]:
# STEP 6 - User interface

import gradio as gr

def chatbot_response(message, history):
    # Use the created RetrievalQA chain to get the answer
    response = qa_chain({"query": message})
    answer = response["result"]
    source_documents = response["source_documents"]

    # Format the response to include the answer and source documents (optional)
    formatted_response = f"{answer}" # You can add source documents here if desired

    return formatted_response

# Create the Gradio interface
iface = gr.ChatInterface(
    fn=chatbot_response,
    title="RAG Chatbot",
    description="Ask questions about Gen-AI and Langchain based on the provided text."
)

# Launch the interface
iface.launch(share=True)